# Center desambiguation using Affilgood

In [1]:
from google.cloud import bigquery
from affilgood import AffilGood
import numpy as np
from tqdm import tqdm
import pandas as pd

In [2]:
PROJECT_ID = 'siris-datasets'
DATASET_ID = 'openalex'

def bg_query(query):
    client = bigquery.Client(project=PROJECT_ID)
    df = client.query(query)
    return df.to_dataframe()

## Get data from the parent centers to see coverage (using OA and OpenAire)

In [3]:
interest_centers = {'BETA' : ['115304662'],
                    'CREAF' : ['123044942', '71999127'],
                    'ICN2' : ['123044942'],
                    'ISGlobal' : ['123044942', '170486558'],
                    'ResearchMar' : ['170486558']}
institutions = list(set(sum(list(interest_centers.values()), [])))
institutions_sql = "(" + ",".join(f"'{i}'" for i in institutions) + ")"
institutions_sql

"('115304662','170486558','123044942','71999127')"

### OpenAlex

In [4]:
sql = f"""
      WITH target_institution AS (SELECT ID
          FROM `{PROJECT_ID}.{DATASET_ID}.institutions`
          WHERE CAST(ID AS STRING) IN {institutions_sql})
          
      SELECT DISTINCT ww.id, ww.DOI, war.raw_affiliation
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      JOIN target_institution wins ON wins.ID = wa.INSTITUTION_ID
      JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships_raw` war ON war.WORK_ID = ww.ID AND war.AUTHOR_ID = wa.AUTHOR_ID
      WHERE ww.PUBLICATION_YEAR > 2020 AND ww.PUBLICATION_YEAR < 2025 
      """
df_raw_affiliations = bg_query(sql).dropna(subset = ['DOI', 'raw_affiliation']).reset_index(drop=True)
df_raw_affiliations = df_raw_affiliations.groupby('raw_affiliation').agg({'id': list, 'DOI': list})
df_raw_affiliations


/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,DOI
raw_affiliation,,
Neurology Department Hospital de Viladecans Barcelona Spain,[4210916112],[10.1111/ane.13586]
"'Acadèmia. És un professor univer-sitari, però -emèrit",[4317930234],[10.7203/puv-esp68-17-converses]
'Arqueologia de Catalunya.,[4386492445],[10.21630/maa.2023.74.03]
"'Arritmies Servei de Cardiologia VHIR, Hospital Universitari Vall d'Hebron, Passeig de la Vall Hebron 119-129, Barcelona 08035, Spain.",[4366541939],[10.4330/wjc.v15.i4.119]
"'Hebron Institut de Recerca (VHIR), Universitat Autònoma de Barcelona (UAB), ARADyAL research network, Instituto de Salud Carlos III (ISCIII), Barcelona, Spain",[3126617176],[10.18176/jiaci.0675]
...,...,...
"�� Universitat de Barcelona ICCUB-IEEC Martí i Franquès 1, 08028, Barcelona",[4387046982],[10.22323/1.417.0017]
"�� Universitat de Barcelona, ICCUB, IEEC-UB, E-08028 Barcelona, Spain","[4366412088, 4385359091]","[10.22323/1.417.0167, 10.22323/1.444.1161]"
"�� University of Barcelona, C/Marti Franques, 1., 08028-Barcelona, Spain",[4324096906],[10.22323/1.420.0005]


In [11]:
affil_good = AffilGood(
    span_separator='',  # Use model-based span identification
    span_model_path='SIRIS-Lab/affilgood-span-multilingual',  # Custom span model
    ner_model_path='SIRIS-Lab/affilgood-NER-multilingual',  # Custom NER model
    entity_linkers=['Whoosh'],  # Use multiple linkers
    return_scores=True,  # Return confidence scores with predictions
    metadata_normalization=True,  # Enable location normalization
    verbose=False,  # Detailed logging
    device='cpu'  # Auto-detect device (CPU or CUDA)
)

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu
Device set to use cpu
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/pydantic/_internal/_generate

WhooshLinker.initialize


In [ ]:
# import concurrent.futures

# def process_with_timeout(affiliations, acom, timeout = 600):
#     """Run affil_good.process(affiliations) with a timeout."""
#     with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
#         future = executor.submit(affil_good.process, affiliations)
#         try:
#             return future.result(timeout=timeout)
#         except concurrent.futures.TimeoutError:
#             print(f"⚠️ Timeout after {timeout} seconds for batch of size {len(affiliations)}. The current batch is the non-processed {acom}")
#             return [None] * len(affiliations)
#         except Exception as e:
#             print(f"⚠️ Error processing batch: {e}")
#             return [None] * len(affiliations)

threshold = 2500
df_raw_affiliations_tmp['affil_good'] = None 
for split in tqdm(np.array_split(df_raw_affiliations_tmp, threshold)): 
    affiliations = split.index.tolist() 
    results = affil_good.process(affiliations) 
    df_raw_affiliations_tmp.loc[split.index, 'affil_good'] = results 
    df_raw_affiliations_tmp.to_csv('../data/interim/affilgood_disambiguation_results_2020_2024_log_1_1.csv')
df_raw_affiliations_tmp

/tmp/ipykernel_9940/2457771986.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_raw_affiliations_tmp['affil_good'] = None
  0%|          | 1/2500 [00:02<1:38:40,  2.37s/it]

In [ ]:
### HAURÉ DE MIRAR QUAN RECUPERO QUE NO ESTIGUI EN EL L'AFILIACIÓ MARE

interest_centers = {'BETA' : [''], # NOT IN OA
                    'CREAF' : ['4210129656', '4401200259'],
                    'ICN2' : ['4210093216'],
                    'ISGlobal' : ['4210148332'],
                    'ResearchMar' : ['4210156109']}
institutions = list(set(sum(list(interest_centers.values()), [])))
institutions_sql = "(" + ",".join(f"'{i}'" for i in institutions) + ")"
institutions_sql

sql = f"""
      WITH target_institution AS (SELECT ID
          FROM `{PROJECT_ID}.{DATASET_ID}.institutions`
          WHERE CAST(ID AS STRING) IN {institutions_sql})
          
      SELECT DISTINCT ww.id, ww.DOI, wins.ID AS INSTITUTION_ID
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      JOIN target_institution wins ON wins.ID = wa.INSTITUTION_ID
      WHERE ww.PUBLICATION_YEAR > 2020 AND ww.PUBLICATION_YEAR < 2025 
      """
df_doi = bg_query(sql).dropna(subset = ['DOI']).reset_index(drop=True)
df_doi


## Benchmark

In [5]:
interest_centers = {'Centro de Investigación y Tecnología Agroalimentaria de Aragón (CITA)' : ['255234318'],
                    'INAGRO - Research & advice in agriculture and horticulture' : [''], # NOT AFFILIATED TO R AND D INSTITUTION
                    'KWR Water Research Institute' : [''], # NOT AFFILIATED TO R AND D INSTITUTION
                    'THE JAMES HUTTON INSTITUTE' : ['177639307'],
                    'SUOMEN YMPARISTOKESKUS (SYKE)' : [''], # NOT AFFILIATED TO R AND D INSTITUTION
                    'UK CENTRE FOR ECOLOGY & HYDROLOGY' : [''], # NOT AFFILIATED TO R AND D INSTITUTION
                    'International Iberian Nanotechnology Laboratory (INL)' : [''], # NOT AFFILIATED TO R AND D INSTITUTION
                    'MESA+ Institute (University of Twente.)' : ['94624287'],
                    'Imdea Nanociencia' : ['63634437'], # NOT TAKING CSIC INTO CONSIDERATION FOR THE AMOUNT OF DATA
                    'The Swiss Tropical and Public Health Institute (SwissTPH)' : ['1850255'],
                    'The London School of Hygiene and Tropical Medicine (LSHTM)' : ['124357947'],
                    'The Liverpool School of Tropical Medicine (LSTM)' : ['146655781'],
                    'FUNDACION PARA LA INVESTIGACION BIOMEDICA DEL HOSPITAL UNIVERSITARIO 12 DE OCTUBRE' : ['121748325'],
                    'CENTRE HOSPITALIER UNIVERSITAIRE DE TOULOUSE' : ['134560555'],
                    'FONDAZIONE IRCCS CA\' GRANDA - OSPEDALE MAGGIORE POLICLINICO' : ['189158943']}
institutions = list(set(sum(list(interest_centers.values()), [])))
institutions_sql = "(" + ",".join(f"'{i}'" for i in filter(None, institutions)) + ")"
institutions_sql

"('124357947','94624287','177639307','134560555','146655781','121748325','63634437','1850255','189158943','255234318')"

In [7]:
sql = f"""
      WITH target_institution AS (SELECT ID
          FROM `{PROJECT_ID}.{DATASET_ID}.institutions`
          WHERE CAST(ID AS STRING) IN {institutions_sql})
          
      SELECT DISTINCT ww.id, ww.DOI, war.raw_affiliation
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      JOIN target_institution wins ON wins.ID = wa.INSTITUTION_ID
      JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships_raw` war ON war.WORK_ID = ww.ID AND war.AUTHOR_ID = wa.AUTHOR_ID
      WHERE ww.PUBLICATION_YEAR > 2020 AND ww.PUBLICATION_YEAR < 2025 
      """
df_raw_affiliations = bg_query(sql).dropna(subset = ['DOI', 'raw_affiliation']).reset_index(drop=True)
df_raw_affiliations = df_raw_affiliations.groupby('raw_affiliation').agg({'id': list, 'DOI': list})
df_raw_affiliations

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,DOI
raw_affiliation,,
Department of Primary Care &amp; Mental Health University of Liverpool Liverpool UK,[4213197585],[10.1111/hsc.13758]
"% de la segregación escolar en España, con muy importantes diferencias entre regiones, llegando al 32,",[4400956310],[10.15366/reice2024.22.3.001]
"% en la Comunidad de Madrid para el alumnado desfavorecido. Estos hallazgos demuestran que los centros privados concertados son, tras la segregación residencial, la mayor fuente de segregación en España y muy superior a la de los países de su entorno. Se concluye así que para limitar la segregación escolar hay que apostar por la escuela pública.",[4400956310],[10.15366/reice2024.22.3.001]
'Alché-Buc Télécom Paris,[4321471920],[10.48550/arxiv.2302.10128]
"'Hebron Barcelona Hospital Campus, CIBER de Enfermedades Respiratorias (CIBERES)",[4394951048],[10.21203/rs.3.rs-4248603/v1]
...,...,...
"�� dpto. Astrofísica, Avda Astrofísico Francisco Sánchez, S/N, 38206 La Laguna (Spain) ��",[4385304712],[10.22323/1.444.0684]
"�� ĲCLab, Université Paris-Saclay, CNRS/IN2P3, 91405 Orsay, France",[4385358370],[10.22323/1.444.0648]
"���� Instituto de Física Teórica UAM-CSIC and Dpto. de Física Teórica C/ Nicolás Cabrera 13-15, Universidad Autónoma de Madrid, Cantoblanco E-28049 Madrid, Spain","[4396675963, 4389925892, 4389912150, 439027270...","[10.22323/1.453.0268, 10.48550/arxiv.2312.1001..."


In [ ]:
### HAURÉ DE MIRAR QUAN RECUPERO QUE NO ESTIGUI EN EL L'AFILIACIÓ MARE

interest_centers = {'Centro de Investigación y Tecnología Agroalimentaria de Aragón (CITA)' : ['4210122787'],
                    'INAGRO - Research & advice in agriculture and horticulture' : ['4210109617'],
                    'KWR Water Research Institute' : ['4210139073'],
                    'THE JAMES HUTTON INSTITUTE' : ['15477984'],
                    'SUOMEN YMPARISTOKESKUS (SYKE)' : ['2800424308'],
                    'UK CENTRE FOR ECOLOGY & HYDROLOGY' : ['4210092773'],
                    'International Iberian Nanotechnology Laboratory (INL)' : ['4210141319'],
                    'MESA+ Institute (University of Twente.)' : [''], # NOT IN OA
                    'Imdea Nanociencia' : ['2802543619'],
                    'The Swiss Tropical and Public Health Institute (SwissTPH)' : ['158937107'],
                    'The London School of Hygiene and Tropical Medicine (LSHTM)' : ['4210089966'],
                    'The Liverpool School of Tropical Medicine (LSTM)' : ['94839184']
                    'FUNDACION PARA LA INVESTIGACION BIOMEDICA DEL HOSPITAL UNIVERSITARIO 12 DE OCTUBRE' : ['4210086614']
                    'CENTRE HOSPITALIER UNIVERSITAIRE DE TOULOUSE' : ['3019448017']
                    'FONDAZIONE IRCCS CA\' GRANDA - OSPEDALE MAGGIORE POLICLINICO' : ['2803066834']}
institutions = list(set(sum(list(interest_centers.values()), [])))
institutions_sql = "(" + ",".join(f"'{i}'" for i in filter(None, institutions)) + ")"
institutions_sql

sql = f"""
      WITH target_institution AS (SELECT ID
          FROM `{PROJECT_ID}.{DATASET_ID}.institutions`
          WHERE CAST(ID AS STRING) IN {institutions_sql})
          
      SELECT DISTINCT ww.id, ww.DOI, wins.ID AS INSTITUTION_ID
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      JOIN target_institution wins ON wins.ID = wa.INSTITUTION_ID
      WHERE ww.PUBLICATION_YEAR > 2020 AND ww.PUBLICATION_YEAR < 2025 
      """
df_doi = bg_query(sql).dropna(subset = ['DOI']).reset_index(drop=True)
df_doi

In [ ]:
# OPEN AIRE